# 🤖 RPA de Extração de Clientes e Disparo para o n8n

## Desafio DIO Santander: Assistente de Investimentos com RPA e IA Generativa

Este notebook executa o processo de **Robotic Process Automation (RPA)**:
1. Faz requisição à página web simulada de clientes (GitHub Pages).
2. Raspa a tabela `#clientes` usando `BeautifulSoup`.
3. Estrutura a lista de investidores (Nome, Email, Saldo, Perfil).
4. Envia os dados estruturados via POST para o Webhook do **n8n**.

**URL da Página:** `https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/`

### 1. Instalação das Dependências

In [ ]:
!pip install requests beautifulsoup4

### 2. Execução do RPA: Raspagem e Estruturação

In [ ]:
import requests
from bs4 import BeautifulSoup

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

# 1) Baixar o conteúdo HTML da página de clientes
print("🌐 Conectando à página de clientes...")
response = requests.get(URL_PAGINA)
response.raise_for_status()
html = response.text

# 2) Parsear a tabela com BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []

for linha in linhas:
    colunas = linha.find_all("td")
    if len(colunas) >= 4:
        cliente = {
            "nome": colunas[0].get_text(strip=True),
            "email": colunas[1].get_text(strip=True),
            "saldo": colunas[2].get_text(strip=True),
            "perfil": colunas[3].get_text(strip=True),
        }
        clientes.append(cliente)

print(f"✅ {len(clientes)} clientes capturados com sucesso:\n")
for idx, c in enumerate(clientes, 1):
    print(f"{idx:02d}. {c['nome']} | {c['email']} | Saldo: {c['saldo']} | Perfil: {c['perfil']}")

### 3. Integração com o n8n via Webhook

Substitua `N8N_WEBHOOK_URL` pela URL do seu Webhook (Production ou Test) gerado no n8n.

In [ ]:
# URL do seu Webhook no n8n (ex: Cloud ou Localhost via túnel ngrok/webhook)
N8N_WEBHOOK_URL = "http://localhost:5678/webhook/clientes"

payload = {"clientes": clientes}

try:
    print(f"🚀 Disparando payload com {len(clientes)} registros para o n8n...")
    res = requests.post(N8N_WEBHOOK_URL, json=payload, timeout=15)
    print(f"Status HTTP: {res.status_code}")
    print(f"Resposta do n8n: {res.text}")
except requests.exceptions.RequestException as err:
    print(f"⚠️ Aviso de Conexão: {err}")
    print("Dica: Certifique-se de que o workflow no n8n está ativo ou com 'Listen for Test Event'.")